# East River Single-Month Forcing Workflow

This notebook walks through a simple, reproducible pipeline for one month:
1. Define month, paths, and basin shapefile
2. Build/select target 4 km grid cells from catchment geometry
3. Acquire daily source data via APIs (PRISM FTP, ERA5-Land cdsapi, Daymet)
4. Harmonize to target grid (Ordinary Kriging with scipy fallback)
5. Build MetSim daily input and disaggregate to hourly
6. Export one monthly SUMMA forcing NetCDF and run checks

In [29]:
from pathlib import Path
import json
import os
import subprocess
from ftplib import FTP
import calendar
import zipfile

import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
from dotenv import load_dotenv

try:
    import cdsapi
    HAS_CDSAPI = True
except Exception:
    HAS_CDSAPI = False

try:
    import pydaymet as daymet
    HAS_PYDAYMET = True
except Exception:
    HAS_PYDAYMET = False

try:
    from pykrige.ok import OrdinaryKriging
    HAS_PYKRIGE = True
except Exception:
    HAS_PYKRIGE = False

from scipy.interpolate import griddata

print('Imports complete. pykrige available:', HAS_PYKRIGE)
print('cdsapi available:', HAS_CDSAPI)
print('pydaymet available:', HAS_PYDAYMET)

Imports complete. pykrige available: True
cdsapi available: True
pydaymet available: True


In [27]:
# Configure one month run
MONTH = '2020-01'
period = pd.Period(MONTH, freq='M')
START = period.start_time
END = period.end_time
YEAR = f'{START.year:04d}'
MM = f'{START.month:02d}'
DAYS_IN_MONTH = calendar.monthrange(START.year, START.month)[1]
DAY_LIST = [f'{d:02d}' for d in range(1, DAYS_IN_MONTH + 1)]

# PRISM daily fields are defined on an 8am-to-8am day window.
DAY_START_HOUR = 8

# MetSim mask and export options.
USE_METSIM_DOMAIN_MASK = True
APPLY_OUTPUT_MASK = True

# Dedicated lightweight MetSim environment executable.
# Leave as-is unless you move/delete the environment.
METSIM_EXE = '/home/dlhogan/miniforge3/envs/metsim-run/bin/ms'

BASIN_ROOT = Path('/scratch/dlhogan/ess-project-data/domain_East_River_lumped')
# Update if your shapefile is located elsewhere.
CATCHMENT_SHP = BASIN_ROOT / 'shapefiles' / 'catchment' / 'East_River_lumped_HRUs_GRUs.shp'
DEM_PATH = BASIN_ROOT / 'attributes' / 'elevation' / 'dem' / 'domain_East_River_lumped_elv.tif'
SUMMA_REFERENCE_PATH = BASIN_ROOT / 'forcing' / 'merged_data' / 'ERA5_merged_201512.nc'

PRISM_FTP_HOST = 'prism.oregonstate.edu'
PRISM_VARIABLES = ['tmin', 'tmax', 'ppt', 'soltotal']
PRISM_REGION = 'us'
PRISM_RES = '4km'
PRISM_DATASET = 'an'

OUT_ROOT = BASIN_ROOT / 'forcing' / 'monthly_workflow'
RAW_DAILY_DIR = OUT_ROOT / 'daily_sources' / MONTH
HARMONIZED_DIR = OUT_ROOT / 'harmonized' / MONTH
METSIM_DIR = OUT_ROOT / 'metsim' / MONTH
FINAL_DIR = OUT_ROOT / 'final_monthly'

METSIM_DOMAIN_PATH = METSIM_DIR / f'metsim_domain_mask_{MONTH}.nc'
METSIM_HOURLY_PATH = METSIM_DIR / f'metsim_hourly_{MONTH}.nc'
SUMMA_FORCING_PATH = FINAL_DIR / f'summa_forcing_{MONTH}.nc'

for p in [RAW_DAILY_DIR, HARMONIZED_DIR, METSIM_DIR, FINAL_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print('Month:', MONTH)
print('Start:', START)
print('End:', END)
print('PRISM day start hour:', DAY_START_HOUR)
print('Use MetSim domain mask:', USE_METSIM_DOMAIN_MASK)
print('Configured MetSim executable:', METSIM_EXE)
print('Catchment shapefile:', CATCHMENT_SHP)
print('DEM path:', DEM_PATH)
print('SUMMA reference:', SUMMA_REFERENCE_PATH)
print('Output root:', OUT_ROOT)
print('PRISM variables:', PRISM_VARIABLES)

Month: 2020-01
Start: 2020-01-01 00:00:00
End: 2020-01-31 23:59:59.999999999
PRISM day start hour: 8
Use MetSim domain mask: True
Configured MetSim executable: /home/dlhogan/miniforge3/envs/metsim-run/bin/ms
Catchment shapefile: /scratch/dlhogan/ess-project-data/domain_East_River_lumped/shapefiles/catchment/East_River_lumped_HRUs_GRUs.shp
DEM path: /scratch/dlhogan/ess-project-data/domain_East_River_lumped/attributes/elevation/dem/domain_East_River_lumped_elv.tif
SUMMA reference: /scratch/dlhogan/ess-project-data/domain_East_River_lumped/forcing/merged_data/ERA5_merged_201512.nc
Output root: /scratch/dlhogan/ess-project-data/domain_East_River_lumped/forcing/monthly_workflow
PRISM variables: ['tmin', 'tmax', 'ppt', 'soltotal']


## Idempotent Workflow

This notebook is designed to be **idempotent**: if output files already exist, they will **not** be re-downloaded or reprocessed. This allows you to safely re-run the notebook to continue where you left off without wasting time or API quota on existing files.

**File existence checks are applied to:**
- ERA5-Land wind data (Step 1c)
- Daymet vapor pressure data (Step 1d)
- MetSim daily input harmonization (Step 4)
- Final monthly SUMMA forcing assembly (Step 6)

If you need to force a re-download, simply delete the output file before re-running.

In [3]:
# Step 1: Use catchment shapefile to identify target grid cells
catchment = gpd.read_file(CATCHMENT_SHP)
catchment = catchment.to_crs('EPSG:4326')
bounds = catchment.total_bounds  # minx, miny, maxx, maxy

# 4 km in degrees is approximate; replace with project-native target grid if available.
dx = 0.0416667
dy = 0.0416667

lons = np.arange(bounds[0] - dx, bounds[2] + dx, dx)
lats = np.arange(bounds[1] - dy, bounds[3] + dy, dy)
lon2d, lat2d = np.meshgrid(lons, lats)
grid_points = gpd.GeoDataFrame(
    {'lon': lon2d.ravel(), 'lat': lat2d.ravel()},
    geometry=gpd.points_from_xy(lon2d.ravel(), lat2d.ravel()),
    crs='EPSG:4326',
)
selected = gpd.sjoin(grid_points, catchment[['geometry']], how='inner', predicate='within')
selected = selected.drop(columns=['index_right']).reset_index(drop=True)

target_cells_path = OUT_ROOT / f'target_gridcells_{MONTH}.geojson'
selected.to_file(target_cells_path, driver='GeoJSON')
print('Selected grid cells:', len(selected))
print('Saved:', target_cells_path)

Selected grid cells: 45
Saved: /scratch/dlhogan/ess-project-data/domain_East_River_lumped/forcing/monthly_workflow/target_gridcells_2020-01.geojson


In [4]:
# Step 1b: Identify PRISM FTP files for this month (tmin/tmax/ppt/soltotal)
def prism_var_dir(var, year):
    return f"/time_series/{PRISM_REGION}/{PRISM_DATASET}/{PRISM_RES}/{var}/daily/{year}"

def list_prism_month_files(var, year, month):
    month_tag = f"{year}{month}"
    with FTP(PRISM_FTP_HOST) as ftp:
        ftp.login()  # anonymous
        ftp.cwd(prism_var_dir(var, year))
        files = ftp.nlst()
    # Keep daily time-series files for the selected month.
    return sorted([f for f in files if month_tag in f and f.endswith('.zip')])

prism_monthly_files = {}
for var in PRISM_VARIABLES:
    try:
        prism_monthly_files[var] = list_prism_month_files(var, YEAR, MM)
    except Exception as exc:
        prism_monthly_files[var] = []
        print(f'Could not list PRISM FTP files for {var}: {exc}')

for var, files in prism_monthly_files.items():
    print(var, 'files found:', len(files))
    if files:
        print('  first:', files[0])
        print('  last :', files[-1])

tmin files found: 31
  first: prism_tmin_us_25m_20200101.zip
  last : prism_tmin_us_25m_20200131.zip
tmax files found: 31
  first: prism_tmax_us_25m_20200101.zip
  last : prism_tmax_us_25m_20200131.zip
ppt files found: 31
  first: prism_ppt_us_25m_20200101.zip
  last : prism_ppt_us_25m_20200131.zip
soltotal files found: 31
  first: prism_soltotal_us_25m_20200101.zip
  last : prism_soltotal_us_25m_20200131.zip


# Section: PRISM processing (chunked, low-memory)

In [5]:
import re
import shutil
import rasterio
from pyproj import Transformer

daily_paths = {
    'prism_tmin': RAW_DAILY_DIR / f'prism_tmin_{MONTH}.nc',
    'prism_tmax': RAW_DAILY_DIR / f'prism_tmax_{MONTH}.nc',
    'prism_precip': RAW_DAILY_DIR / f'prism_precip_{MONTH}.nc',
    'prism_soltotal': RAW_DAILY_DIR / f'prism_soltotal_{MONTH}.nc',
    'era5land_wind': RAW_DAILY_DIR / f'era5land_wind_{MONTH}.nc',
    'daymet_vp': RAW_DAILY_DIR / f'daymet_vp_{MONTH}.nc',
}

PRISM_ZIP_DIR = RAW_DAILY_DIR / 'prism_zips'
PRISM_TMP_DIR = RAW_DAILY_DIR / 'prism_tmp'
PRISM_ZIP_DIR.mkdir(parents=True, exist_ok=True)
PRISM_TMP_DIR.mkdir(parents=True, exist_ok=True)


def _extract_yyyymmdd(text):
    m = re.search(r'(\d{8})', text)
    return m.group(1) if m else None


def _find_raster_for_date(search_dir, yyyymmdd):
    for p in sorted(search_dir.rglob('*')):
        if not p.is_file():
            continue
        name = p.name.lower()
        if yyyymmdd not in name:
            continue
        if name.endswith('.bil') or name.endswith('.tif') or name.endswith('.tiff'):
            return p
    return None


def _sample_raster_to_selected(raster_path, selected_gdf):
    with rasterio.open(raster_path) as src:
        lons = selected_gdf['lon'].to_numpy()
        lats = selected_gdf['lat'].to_numpy()

        if src.crs is not None and str(src.crs).upper() not in {'EPSG:4326', 'OGC:CRS84'}:
            transformer = Transformer.from_crs('EPSG:4326', src.crs, always_xy=True)
            xs, ys = transformer.transform(lons, lats)
            pts = list(zip(xs, ys))
        else:
            pts = list(zip(lons, lats))

        vals = np.array([v[0] for v in src.sample(pts)], dtype=float)
        nodata = src.nodata
        if nodata is not None:
            vals = np.where(vals == nodata, np.nan, vals)
        return vals


def _download_prism_zip_if_missing(var, fname, zip_var_dir):
    zpath = zip_var_dir / fname
    if zpath.exists():
        return zpath
    with FTP(PRISM_FTP_HOST) as ftp:
        ftp.login()
        ftp.cwd(prism_var_dir(var, YEAR))
        print(f'  {var}: downloading {fname}')
        with open(zpath, 'wb') as f:
            ftp.retrbinary(f'RETR {fname}', f.write)
    return zpath


def _process_prism_zip_to_chunk(var, zpath, yyyymmdd, selected_gdf, lat_vals, lon_vals, chunk_path, tmp_extract_root):
    extract_dir = tmp_extract_root / zpath.stem
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True, exist_ok=True)

    try:
        with zipfile.ZipFile(zpath, 'r') as zf:
            zf.extractall(extract_dir)

        raster_path = _find_raster_for_date(extract_dir, yyyymmdd)
        if raster_path is None:
            print(f'  {var}: raster not found after unzip for {yyyymmdd}')
            return False

        sampled = _sample_raster_to_selected(raster_path, selected_gdf)
        lat_to_i = {v: i for i, v in enumerate(lat_vals)}
        lon_to_j = {v: j for j, v in enumerate(lon_vals)}
        grid = np.full((len(lat_vals), len(lon_vals)), np.nan, dtype=float)

        for k, val in enumerate(sampled):
            lat = selected.iloc[k]['lat']
            lon = selected.iloc[k]['lon']
            i = lat_to_i.get(lat)
            j = lon_to_j.get(lon)
            if i is not None and j is not None:
                grid[i, j] = val

        out_name = {
            'tmin': 'prism_tmin',
            'tmax': 'prism_tmax',
            'ppt': 'prism_precip',
            'soltotal': 'prism_soltotal',
        }[var]

        ds_day = xr.Dataset(
            {out_name: (('time', 'latitude', 'longitude'), grid[np.newaxis, :, :])},
            coords={
                'time': [pd.to_datetime(yyyymmdd, format='%Y%m%d')],
                'latitude': lat_vals,
                'longitude': lon_vals,
            },
        )
        ds_day.to_netcdf(chunk_path)
        return True
    finally:
        if extract_dir.exists():
            shutil.rmtree(extract_dir)


def _build_prism_monthly(var, output_path):
    if output_path.exists():
        print(f'  {var}: output exists, skipping build ({output_path.name})')
        return

    if var not in prism_monthly_files or not prism_monthly_files[var]:
        try:
            prism_monthly_files[var] = list_prism_month_files(var, YEAR, MM)
        except Exception as exc:
            print(f'  {var}: failed listing PRISM files: {exc}')
            prism_monthly_files[var] = []

    month_files = prism_monthly_files.get(var, [])
    if not month_files:
        print(f'  {var}: no PRISM files found for {MONTH}')
        return

    zip_var_dir = PRISM_ZIP_DIR / var
    tmp_var_dir = PRISM_TMP_DIR / var
    tmp_extract_root = tmp_var_dir / 'extract'
    tmp_chunks_dir = tmp_var_dir / 'chunks'
    zip_var_dir.mkdir(parents=True, exist_ok=True)
    tmp_extract_root.mkdir(parents=True, exist_ok=True)
    tmp_chunks_dir.mkdir(parents=True, exist_ok=True)

    lat_vals = np.sort(selected['lat'].unique())
    lon_vals = np.sort(selected['lon'].unique())

    for fname in month_files:
        yyyymmdd = _extract_yyyymmdd(fname)
        if yyyymmdd is None:
            continue
        chunk_path = tmp_chunks_dir / f'{var}_{yyyymmdd}.nc'
        if chunk_path.exists():
            continue

        zpath = _download_prism_zip_if_missing(var, fname, zip_var_dir)
        ok = _process_prism_zip_to_chunk(var, zpath, yyyymmdd, selected, lat_vals, lon_vals, chunk_path, tmp_extract_root)
        if ok:
            print(f'  {var}: chunk ready for {yyyymmdd}')

    chunk_files = sorted(tmp_chunks_dir.glob(f'{var}_*.nc'))
    if not chunk_files:
        print(f'  {var}: no chunk files generated')
        return

    ds_parts = []
    for fp in chunk_files:
        with xr.open_dataset(fp) as dsi:
            ds_parts.append(dsi.load())

    ds_out = xr.concat(ds_parts, dim='time').sortby('time')

    units = {
        'tmin': 'degC',
        'tmax': 'degC',
        'ppt': 'mm',
        'soltotal': 'MJ m-2 day-1',
    }[var]
    out_name = list(ds_out.data_vars)[0]
    ds_out[out_name].attrs['units'] = units
    ds_out[out_name].attrs['source'] = f'PRISM {var} daily 4km'

    ds_out.to_netcdf(output_path)
    print(f'  {var}: wrote {output_path}')

    if tmp_var_dir.exists():
        shutil.rmtree(tmp_var_dir)


_build_prism_monthly('tmin', daily_paths['prism_tmin'])
_build_prism_monthly('tmax', daily_paths['prism_tmax'])
_build_prism_monthly('ppt', daily_paths['prism_precip'])
_build_prism_monthly('soltotal', daily_paths['prism_soltotal'])

  tmin: output exists, skipping build (prism_tmin_2020-01.nc)
  tmax: output exists, skipping build (prism_tmax_2020-01.nc)
  ppt: output exists, skipping build (prism_precip_2020-01.nc)
  soltotal: output exists, skipping build (prism_soltotal_2020-01.nc)


# Section: Daymet
Build Daymet request for vapor pressure (vp)

In [6]:

daymet_vp_out = RAW_DAILY_DIR / f'daymet_vp_{MONTH}.nc'
print('Daymet output file:', daymet_vp_out)

from shapely.geometry import Polygon
polygon = Polygon(catchment.geometry.iloc[0].exterior.coords)

# Optional Earthdata credentials (only used by endpoints that require auth)
load_dotenv()
username = os.getenv('EARTHDATA_USERNAME')
password = os.getenv('EARTHDATA_PASSWORD')
if username and password:
    os.environ['EARTHDATA_USERNAME'] = username
    os.environ['EARTHDATA_PASSWORD'] = password


def _daymet_points_to_grid(ds_in, target_points, start_dt, end_dt):
    """Convert Daymet fallback output from (time, id) to (time, latitude, longitude)."""
    ds = ds_in.copy()

    if 'time' in ds.coords:
        ds = ds.sel(time=slice(pd.Timestamp(start_dt).normalize(), pd.Timestamp(end_dt).normalize()))

    if 'id' not in ds.dims:
        rename_map = {}
        if 'y' in ds.dims:
            rename_map['y'] = 'latitude'
        if 'x' in ds.dims:
            rename_map['x'] = 'longitude'
        if rename_map:
            ds = ds.rename(rename_map)
        return ds

    n_id = ds.sizes['id']
    if n_id != len(target_points):
        raise ValueError(f'Daymet id length ({n_id}) does not match target cell count ({len(target_points)}).')

    lon_vals = target_points['lon'].to_numpy()
    lat_vals = target_points['lat'].to_numpy()

    ds = ds.assign_coords(longitude=('id', lon_vals), latitude=('id', lat_vals))
    ds = ds.set_index(id=['latitude', 'longitude']).unstack('id')

    if 'time' in ds.dims:
        ds = ds.transpose('time', 'latitude', 'longitude')

    ds = ds.sortby('latitude').sortby('longitude')
    return ds


if HAS_PYDAYMET:
    if daymet_vp_out.exists():
        print('  (already exists, opening and normalizing grid if needed)')
        with xr.open_dataset(daymet_vp_out) as ds_existing:
            ds_daymet = ds_existing.load()
    else:
        try:
            ds_daymet = daymet.get_bygeom(
                polygon,
                dates=(START.date().isoformat(), END.date().isoformat()),
                variables=['vp'],
                time_scale='daily',
            )
            print('Daymet source: get_bygeom (NCSS)')
        except Exception as exc:
            print(f'get_bygeom failed: {exc}')
            print('Falling back to Daymet single-pixel API via get_bycoords.')

            coords = list(zip(selected['lon'].to_numpy(), selected['lat'].to_numpy()))
            ds_daymet = daymet.get_bycoords(
                coords=coords,
                dates=(START.date().isoformat(), END.date().isoformat()),
                variables=['vp'],
                time_scale='daily',
                to_xarray=True,
            )
            print('Daymet source: get_bycoords (single-pixel API)')

    ds_daymet = _daymet_points_to_grid(ds_daymet, selected, START, END)

    tmp_out = daymet_vp_out.with_name(f'{daymet_vp_out.stem}.tmp.nc')
    if tmp_out.exists():
        tmp_out.unlink()
    ds_daymet.to_netcdf(tmp_out)
    os.replace(tmp_out, daymet_vp_out)

    print('Saved normalized Daymet dataset:', daymet_vp_out)
    print('Daymet dims:', dict(ds_daymet.sizes))
    print('Daymet coords:', list(ds_daymet.coords))
else:
    print('pydaymet is not available in this environment.')

Daymet output file: /scratch/dlhogan/ess-project-data/domain_East_River_lumped/forcing/monthly_workflow/daily_sources/2020-01/daymet_vp_2020-01.nc
  (already exists, opening and normalizing grid if needed)
Saved normalized Daymet dataset: /scratch/dlhogan/ess-project-data/domain_East_River_lumped/forcing/monthly_workflow/daily_sources/2020-01/daymet_vp_2020-01.nc
Daymet dims: {'latitude': 8, 'longitude': 8, 'time': 31}
Daymet coords: ['latitude', 'longitude', 'time']


# Section: ERA5
Build ERA5-Land cdsapi request template for wind components

In [7]:

era5_request = {
    'variable': ['10m_u_component_of_wind', '10m_v_component_of_wind'],
    'year': YEAR,
    'month': MM,
    'day': DAY_LIST,
    'time': ['00:00', '06:00', '12:00', '18:00'],
    'format': 'netcdf',
    # [N, W, S, E] from catchment bounds in EPSG:4326
    'area': [float(bounds[3]), float(bounds[0]), float(bounds[1]), float(bounds[2])],
}
era5_out = RAW_DAILY_DIR / f'era5land_u_v_{MONTH}.nc'
era5_unzipped_out = RAW_DAILY_DIR / f'era5land_u_v_{MONTH}_unzipped.nc'
print('ERA5-Land request template ready')
print('Output file:', era5_out)
print('Request keys:', sorted(era5_request.keys()))

if HAS_CDSAPI:
    if era5_out.exists() or era5_unzipped_out.exists():
        print('  (already exists, skipping download)')
    else:
        c = cdsapi.Client()
        c.retrieve('reanalysis-era5-land', era5_request, str(era5_out))

if era5_out.exists() and zipfile.is_zipfile(era5_out):
    with zipfile.ZipFile(era5_out, 'r') as zf:
        nc_members = [m for m in zf.namelist() if m.lower().endswith('.nc')]
        if not nc_members:
            raise RuntimeError(f'No .nc file found inside {era5_out}')
        with zf.open(nc_members[0]) as src, open(era5_unzipped_out, 'wb') as dst:
            dst.write(src.read())
    era5_out.unlink()
    print('Unzipped ERA5 netcdf to:', era5_unzipped_out)
    print('Removed ZIP archive:', era5_out)
    era5_out = era5_unzipped_out
elif era5_unzipped_out.exists():
    era5_out = era5_unzipped_out

ERA5-Land request template ready
Output file: /scratch/dlhogan/ess-project-data/domain_East_River_lumped/forcing/monthly_workflow/daily_sources/2020-01/era5land_u_v_2020-01.nc
Request keys: ['area', 'day', 'format', 'month', 'time', 'variable', 'year']
  (already exists, skipping download)


In [8]:
# Quick ERA5 read check
# If the downloaded file is still a zip archive, unpack and open the first .nc.
import zipfile

era5_read_path = era5_out
if era5_read_path.exists() and zipfile.is_zipfile(era5_read_path):
    unzip_target = RAW_DAILY_DIR / f'era5land_u_v_{MONTH}_unzipped.nc'
    if not unzip_target.exists():
        with zipfile.ZipFile(era5_read_path, 'r') as zf:
            nc_members = [m for m in zf.namelist() if m.lower().endswith('.nc')]
            if not nc_members:
                raise RuntimeError(f'No .nc file found inside {era5_read_path}')
            with zf.open(nc_members[0]) as src, open(unzip_target, 'wb') as dst:
                dst.write(src.read())
    era5_read_path = unzip_target

# Explicit engine avoids backend guessing issues.
ds_era5 = xr.open_dataset(era5_read_path, engine='netcdf4')
print('ERA5 read path:', era5_read_path)


ERA5 read path: /scratch/dlhogan/ess-project-data/domain_East_River_lumped/forcing/monthly_workflow/daily_sources/2020-01/era5land_u_v_2020-01_unzipped.nc


# Section: Data resampling / downscaling
Harmonize a source field to target cells with kriging fallback

In [9]:

def interpolate_to_target(x_obs, y_obs, z_obs, x_tgt, y_tgt):
    if HAS_PYKRIGE and len(z_obs) >= 8:
        try:
            ok = OrdinaryKriging(
                x_obs, y_obs, z_obs,
                variogram_model='spherical',
                verbose=False,
                enable_plotting=False,
            )
            z_tgt, _ = ok.execute('points', x_tgt, y_tgt)
            return np.asarray(z_tgt), 'ordinary_kriging'
        except Exception:
            pass

    # scipy fallback
    z_tgt = griddata((x_obs, y_obs), z_obs, (x_tgt, y_tgt), method='linear')
    if np.any(np.isnan(z_tgt)):
        z_near = griddata((x_obs, y_obs), z_obs, (x_tgt, y_tgt), method='nearest')
        z_tgt = np.where(np.isnan(z_tgt), z_near, z_tgt)
    return z_tgt, 'scipy_griddata'

print('Interpolation helper ready.')

Interpolation helper ready.


# Section: Aggregation
Build MetSim daily input dataset for this month

Expected variables at daily step (example names):
- t_min, t_max, precip, wind, shortwave(from PRISM soltotal), vapor_pressure(from Daymet)

Keep this monthly and deterministic.

In [39]:
metsim_daily_path = METSIM_DIR / f'metsim_daily_input_{MONTH}.nc'
needs_rebuild = False
if metsim_daily_path.exists():
    with xr.open_dataset(metsim_daily_path) as ds_existing_daily:
        ds_metsim_daily = ds_existing_daily.load()
    t_units_existing = str(ds_metsim_daily['t_min'].attrs.get('units', '')).lower()
    p_units_existing = str(ds_metsim_daily['precip'].attrs.get('units', '')).lower()
    # Rebuild if an older run wrote SUMMA-style units into the MetSim daily input.
    if ('k' in t_units_existing) or ('mm s' in p_units_existing) or ('mm/s' in p_units_existing):
        needs_rebuild = True
        print('Existing MetSim daily input uses non-MetSim units; rebuilding:', metsim_daily_path)
else:
    needs_rebuild = True

if not needs_rebuild:
    print(f'MetSim daily input already exists and is unit-compatible: {metsim_daily_path}')
else:
    print(f'Write monthly MetSim daily input to: {metsim_daily_path}')

    def _open_nc(path):
        if not path.exists():
            raise FileNotFoundError(f'Missing required input: {path}')
        try:
            return xr.open_dataset(path, engine='netcdf4')
        except Exception:
            return xr.open_dataset(path)

    def _lat_lon_names(obj):
        lat_name = 'latitude' if 'latitude' in obj.coords else ('lat' if 'lat' in obj.coords else None)
        lon_name = 'longitude' if 'longitude' in obj.coords else ('lon' if 'lon' in obj.coords else None)
        return lat_name, lon_name

    def _time_name(obj):
        for name in ['time', 'valid_time'] + list(obj.coords):
            if name in obj.coords and np.issubdtype(obj[name].dtype, np.datetime64):
                return name
        for name in obj.dims:
            if name.lower() == 'time':
                return name
        return None

    def _to_target_grid(da_2d, target_lat, target_lon):
        lat_name, lon_name = _lat_lon_names(da_2d)
        if lat_name is None or lon_name is None:
            raise ValueError('Source field is missing lat/lon coordinates for interpolation.')

        src_lat = da_2d[lat_name].values
        src_lon = da_2d[lon_name].values
        src_vals = da_2d.values

        src_lon_2d, src_lat_2d = np.meshgrid(src_lon, src_lat)
        x_obs = src_lon_2d.ravel()
        y_obs = src_lat_2d.ravel()
        z_obs = src_vals.ravel()

        valid = np.isfinite(z_obs)
        if valid.sum() < 3:
            raise ValueError('Not enough valid source points for interpolation.')

        x_tgt_2d, y_tgt_2d = np.meshgrid(target_lon, target_lat)
        z_tgt, method = interpolate_to_target(
            x_obs[valid],
            y_obs[valid],
            z_obs[valid],
            x_tgt_2d.ravel(),
            y_tgt_2d.ravel(),
        )
        z_grid = np.asarray(z_tgt).reshape(len(target_lat), len(target_lon))
        return xr.DataArray(
            z_grid,
            dims=('latitude', 'longitude'),
            coords={'latitude': target_lat, 'longitude': target_lon},
            attrs={'interpolation': method},
        )

    with _open_nc(daily_paths['prism_tmin']) as ds_tmin, \
         _open_nc(daily_paths['prism_tmax']) as ds_tmax, \
         _open_nc(daily_paths['prism_precip']) as ds_ppt, \
         _open_nc(daily_paths['prism_soltotal']) as ds_swrad, \
         _open_nc(daymet_vp_out) as ds_daymet, \
         _open_nc(era5_out) as ds_era5:

        target_lat = np.sort(selected['lat'].unique())
        target_lon = np.sort(selected['lon'].unique())

        era5_lat_name, era5_lon_name = ('latitude' if 'latitude' in ds_era5.coords else 'lat'), ('longitude' if 'longitude' in ds_era5.coords else 'lon')
        if era5_lat_name is None or era5_lon_name is None:
            raise ValueError('ERA5 dataset is missing latitude/longitude coordinates.')

        if float(ds_era5[era5_lon_name].max()) > 180.0:
            ds_era5 = ds_era5.assign_coords(
                {era5_lon_name: (((ds_era5[era5_lon_name] + 180) % 360) - 180)}
            ).sortby(era5_lon_name)

        era5_time = _time_name(ds_era5)
        if era5_time is None:
            raise ValueError('ERA5 dataset is missing a datetime-like time coordinate.')
        if era5_time != 'time':
            ds_era5 = ds_era5.rename({era5_time: 'time'})

        u_candidates = ['u10', '10m_u_component_of_wind']
        v_candidates = ['v10', '10m_v_component_of_wind']
        u_name = next((v for v in u_candidates if v in ds_era5.data_vars), None)
        v_name = next((v for v in v_candidates if v in ds_era5.data_vars), None)
        if u_name is None or v_name is None:
            raise ValueError(f'Could not find ERA5 wind components. Found: {list(ds_era5.data_vars)}')

        da_ws = np.hypot(ds_era5[u_name], ds_era5[v_name])
        da_ws.name = 'wind'

        ws_interp_parts = []
        for t in da_ws['time'].values:
            da2d = da_ws.sel(time=t)
            ws_interp = _to_target_grid(da2d, target_lat, target_lon).expand_dims(time=[pd.Timestamp(t)])
            ws_interp_parts.append(ws_interp)

        da_ws_target = xr.concat(ws_interp_parts, dim='time').sortby('time')
        da_ws_daily = da_ws_target.resample(time='1D').mean()

        vp_name = 'vp' if 'vp' in ds_daymet.data_vars else list(ds_daymet.data_vars)[0]

        tmin = ds_tmin['prism_tmin']
        tmax = ds_tmax['prism_tmax']
        precip = ds_ppt['prism_precip']
        shortwave = ds_swrad['prism_soltotal']
        vapor_pressure = ds_daymet[vp_name]

        month_slice = slice(pd.Timestamp(START).normalize(), pd.Timestamp(END).normalize())
        tmin = tmin.sel(time=month_slice)
        tmax = tmax.sel(time=month_slice)
        precip = precip.sel(time=month_slice)
        shortwave = shortwave.sel(time=month_slice)
        vapor_pressure = vapor_pressure.sel(time=month_slice)
        da_ws_daily = da_ws_daily.sel(time=month_slice)

        tmin, tmax, precip, shortwave, vapor_pressure, da_ws_daily = xr.align(
            tmin, tmax, precip, shortwave, vapor_pressure, da_ws_daily, join='inner'
        )

        # MetSim daily inputs must be physical MetSim units (not SUMMA units):
        # t_min/t_max in degC, precip in mm day-1, shortwave in W m-2 (daily mean).
        shortwave = shortwave * (1.0e6 / 86400.0)

        ds_metsim_daily = xr.Dataset(
            data_vars={
                't_min': tmin,
                't_max': tmax,
                'precip': precip,
                'shortwave': shortwave,
                'vapor_pressure': vapor_pressure,
                'wind': da_ws_daily,
            },
            coords={
                'time': tmin['time'],
                'latitude': tmin['latitude'],
                'longitude': tmin['longitude'],
            },
        )

        shifted_time = pd.DatetimeIndex(ds_metsim_daily['time'].values).normalize() + pd.Timedelta(hours=DAY_START_HOUR)
        ds_metsim_daily = ds_metsim_daily.assign_coords(time=shifted_time)
        ds_metsim_daily = ds_metsim_daily.sortby('time')
        ds_metsim_daily.attrs['day_definition'] = f'{DAY_START_HOUR:02d}:00-to-{DAY_START_HOUR:02d}:00 local day (PRISM convention)'
        ds_metsim_daily.attrs['unit_conversion'] = 'shortwave: MJ m-2 day-1 -> W m-2; t_min/t_max kept as degC; precip kept as mm day-1'

        ds_metsim_daily['t_min'].attrs.update({'units': 'C', 'source': 'PRISM tmin'})
        ds_metsim_daily['t_max'].attrs.update({'units': 'C', 'source': 'PRISM tmax'})
        ds_metsim_daily['precip'].attrs.update({'units': 'mm day-1', 'source': 'PRISM ppt'})
        ds_metsim_daily['shortwave'].attrs.update({'units': 'W m-2', 'source': 'PRISM soltotal converted from MJ m-2 day-1'})
        ds_metsim_daily['vapor_pressure'].attrs.update({'units': 'Pa', 'source': 'Daymet vp'})
        ds_metsim_daily['wind'].attrs.update({'units': 'm s-1', 'source': 'ERA5-Land u10/v10 daily mean'})

        tmp_daily = metsim_daily_path.with_name(f'{metsim_daily_path.stem}.tmp.nc')
        if tmp_daily.exists():
            tmp_daily.unlink()
        ds_metsim_daily.to_netcdf(tmp_daily)
        os.replace(tmp_daily, metsim_daily_path)
        print('Wrote:', metsim_daily_path)
        print('Dims:', dict(ds_metsim_daily.sizes))
        print('Vars:', list(ds_metsim_daily.data_vars))
        print('Daily forcing time span:', pd.Timestamp(ds_metsim_daily.time.values[0]), 'to', pd.Timestamp(ds_metsim_daily.time.values[-1]))

target_lat = np.asarray(ds_metsim_daily['latitude'].values)
target_lon = np.asarray(ds_metsim_daily['longitude'].values)
tgt_lon_2d, tgt_lat_2d = np.meshgrid(target_lon, target_lat)
with rasterio.open(DEM_PATH) as dem_src:
    pts = list(zip(tgt_lon_2d.ravel(), tgt_lat_2d.ravel()))
    elev_vals = np.array([v[0] for v in dem_src.sample(pts)], dtype=float).reshape(tgt_lat_2d.shape)
    if dem_src.nodata is not None:
        elev_vals = np.where(elev_vals == dem_src.nodata, np.nan, elev_vals)

if np.isnan(elev_vals).any():
    src_mask = np.isfinite(elev_vals)
    z_tgt, _ = interpolate_to_target(
        tgt_lon_2d[src_mask],
        tgt_lat_2d[src_mask],
        elev_vals[src_mask],
        tgt_lon_2d.ravel(),
        tgt_lat_2d.ravel(),
    )
    elev_vals = np.asarray(z_tgt).reshape(tgt_lat_2d.shape)

mask2d = np.isfinite(ds_metsim_daily['t_min'].isel(time=0)).astype(np.int8).values
elev_vals = np.where(mask2d > 0, elev_vals, np.nan)

lon_grid = np.broadcast_to(tgt_lon_2d, mask2d.shape).astype(np.float32)
lat_grid = np.broadcast_to(tgt_lat_2d, mask2d.shape).astype(np.float32)

ds_domain = xr.Dataset(
    {
        'mask': (('latitude', 'longitude'), mask2d),
        'elev': (('latitude', 'longitude'), elev_vals.astype(np.float32)),
        'lat': (('latitude', 'longitude'), lat_grid),
        'lon': (('latitude', 'longitude'), lon_grid),
    },
    coords={
        'latitude': target_lat,
        'longitude': target_lon,
    },
)
ds_domain['mask'].attrs.update({'units': '1', 'description': '1=in-basin, 0=outside'})
ds_domain['elev'].attrs.update({'units': 'm', 'source': str(DEM_PATH)})
ds_domain['lat'].attrs.update({'units': 'degrees_north'})
ds_domain['lon'].attrs.update({'units': 'degrees_east'})
ds_domain.to_netcdf(METSIM_DOMAIN_PATH)
print('Wrote domain with mask+elev:', METSIM_DOMAIN_PATH)

def inspect_coord_names(nc_path):
    if not nc_path.exists():
        return {'exists': False}
    try:
        ds = xr.open_dataset(nc_path, engine='netcdf4')
    except Exception:
        ds = xr.open_dataset(nc_path)
    return {'exists': True, 'dims': list(ds.dims), 'coords': list(ds.coords), 'vars': list(ds.data_vars)}

print('\nCoordinate audit (latitude/longitude consistency):')
audit_targets = {
    'era5': era5_out,
    'daymet': daymet_vp_out,
    'prism_tmin': daily_paths['prism_tmin'],
    'prism_tmax': daily_paths['prism_tmax'],
    'prism_precip': daily_paths['prism_precip'],
    'prism_soltotal': daily_paths['prism_soltotal'],
    'metsim_domain': METSIM_DOMAIN_PATH,
    'metsim_daily': metsim_daily_path,
}
for name, path in audit_targets.items():
    info = inspect_coord_names(path)
    if not info['exists']:
        print(f' - {name}: missing ({path})')
    else:
        print(f" - {name}: dims={info['dims']} coords={info['coords']} vars={info['vars']}")

MetSim daily input already exists and is unit-compatible: /scratch/dlhogan/ess-project-data/domain_East_River_lumped/forcing/monthly_workflow/metsim/2020-01/metsim_daily_input_2020-01.nc
Wrote domain with mask+elev: /scratch/dlhogan/ess-project-data/domain_East_River_lumped/forcing/monthly_workflow/metsim/2020-01/metsim_domain_mask_2020-01.nc

Coordinate audit (latitude/longitude consistency):
 - era5: dims=['valid_time', 'latitude', 'longitude'] coords=['number', 'valid_time', 'latitude', 'longitude', 'expver'] vars=['u10', 'v10']
 - daymet: dims=['latitude', 'longitude', 'time'] coords=['latitude', 'longitude', 'time'] vars=['vp']
 - prism_tmin: dims=['time', 'latitude', 'longitude'] coords=['time', 'latitude', 'longitude'] vars=['prism_tmin']
 - prism_tmax: dims=['time', 'latitude', 'longitude'] coords=['time', 'latitude', 'longitude'] vars=['prism_tmax']
 - prism_precip: dims=['time', 'latitude', 'longitude'] coords=['time', 'latitude', 'longitude'] vars=['prism_precip']
 - prism_s

# Section: MetSim
Run MetSim to disaggregate to hourly and derive pressure/longwave

You can generate this config from Python or keep a template in the repo.

In [41]:
# MetSim expects start/stop within the daily forcing index range.
# The daily input already carries PRISM-style 08:00 timestamps.
import shutil

metsim_start = pd.Timestamp(START).normalize()
metsim_stop = pd.Timestamp(END).normalize()

# Always rebuild a deterministic 90-day state file from current daily inputs.
# This avoids stale state files after unit-conversion changes upstream.
metsim_state_path = METSIM_DIR / f'metsim_state_{MONTH}.nc'
with xr.open_dataset(metsim_daily_path) as ds_daily_in:
    ds_daily = ds_daily_in.load()

state_dates = pd.date_range(end=metsim_start - pd.Timedelta(days=1), periods=90, freq='D')
seed_tmin = ds_daily['t_min'].isel(time=0).astype(np.float32)
seed_tmax = ds_daily['t_max'].isel(time=0).astype(np.float32)
seed_prec = ds_daily['precip'].isel(time=0).astype(np.float32)

state_tmin = xr.concat([seed_tmin] * len(state_dates), dim='time').assign_coords(time=state_dates)
state_tmax = xr.concat([seed_tmax] * len(state_dates), dim='time').assign_coords(time=state_dates)
state_prec = xr.concat([seed_prec] * len(state_dates), dim='time').assign_coords(time=state_dates)

ds_state = xr.Dataset(
    {
        't_min': state_tmin,
        't_max': state_tmax,
        'precip': state_prec,
    },
    coords={
        'time': state_dates,
        'latitude': ds_daily['latitude'],
        'longitude': ds_daily['longitude'],
    },
)
ds_state['t_min'].attrs['units'] = ds_daily['t_min'].attrs.get('units', 'C')
ds_state['t_max'].attrs['units'] = ds_daily['t_max'].attrs.get('units', 'C')
ds_state['precip'].attrs['units'] = ds_daily['precip'].attrs.get('units', 'mm day-1')
ds_state.to_netcdf(metsim_state_path)
print('Wrote MetSim state file:', metsim_state_path)

# MetSim CLI expects INI/YAML config, not JSON.
# Use canonical mappings so our variable names are explicit and reproducible.
config_path = METSIM_DIR / f'metsim_config_{MONTH}.ini'
config_lines = [
    '[MetSim]',
    f'start = {metsim_start:%Y-%m-%d}',
    f'stop = {metsim_stop:%Y-%m-%d}',
    'time_step = 60',
    f'forcing = {metsim_daily_path}',
    f'state = {metsim_state_path}',
    f'out_dir = {METSIM_DIR}',
    'forcing_fmt = netcdf',
    'lw_type = prata',
    f'out_prefix = metsim_hourly_{MONTH}',
    '',
    '[forcing_vars]',
    't_min = t_min',
    't_max = t_max',
    'prec = precip',
    'wind = wind',
    'shortwave = shortwave',
    'vapor_pressure = vapor_pressure',
    '',
    '[out_vars]',
    'temp = airtemp',
    'prec = precip',
    'shortwave = SWRadAtm',
    'longwave = LWRadAtm',
    'vapor_pressure = vapor_pressure',
    'wind = wind',
    'air_pressure = airpres',
    'spec_humid = spechum',
    '',
    '[domain_vars]',
    'mask = mask',
    'elev = elev',
    'lat = lat',
    'lon = lon',
    '',
    '[state_vars]',
    't_min = t_min',
    't_max = t_max',
    'prec = precip',
    '',
    '[chunks]',
    '',
]

if USE_METSIM_DOMAIN_MASK and METSIM_DOMAIN_PATH.exists():
    insert_at = config_lines.index('')
    config_lines.insert(insert_at, f'domain = {METSIM_DOMAIN_PATH}')

config_text = '\n'.join(config_lines).rstrip() + '\n'
config_path.write_text(config_text, encoding='utf-8')

# Resolve MetSim executable in priority order:
# 1) METSIM_EXE from config cell
# 2) METSIM_EXE environment variable
# 3) `ms` on PATH
candidate_cmds = [
    METSIM_EXE,
    os.getenv('METSIM_EXE'),
    shutil.which('ms'),
]
metsim_exe = next((c for c in candidate_cmds if c and Path(c).exists()), None)
if metsim_exe is None:
    raise RuntimeError(
        'Could not locate MetSim executable `ms`. '
        'Expected at METSIM_EXE or on PATH. '
        'Create env with: mamba create -n metsim-run -c conda-forge python=3.10 metsim -y'
    )

metsim_cmd = [str(metsim_exe), str(config_path)]

print('Saved MetSim config:', config_path)
print('Configured start/stop:', metsim_start, metsim_stop)
if USE_METSIM_DOMAIN_MASK and METSIM_DOMAIN_PATH.exists():
    print('Using domain file:', METSIM_DOMAIN_PATH)
else:
    print('Domain file not set in config (file missing or USE_METSIM_DOMAIN_MASK=False).')
print('MetSim command:', ' '.join(metsim_cmd))

proc = subprocess.run(metsim_cmd, text=True, capture_output=True)
if proc.stdout:
    print(proc.stdout)
if proc.returncode != 0:
    if proc.stderr:
        print(proc.stderr)
    raise RuntimeError(f'MetSim failed with exit code {proc.returncode}')

# Always sync canonical path to the newest MetSim hourly file.
produced_nc = sorted(
    METSIM_DIR.glob(f'metsim_hourly_{MONTH}*.nc'),
    key=lambda p: p.stat().st_mtime,
    reverse=True,
 )
if not produced_nc:
    raise FileNotFoundError(f'No MetSim hourly outputs found in {METSIM_DIR}')

latest = produced_nc[0]
print('Most recent NetCDF in output dir:', latest)
if latest != METSIM_HOURLY_PATH:
    shutil.copy2(latest, METSIM_HOURLY_PATH)
    print('Copied latest NetCDF to canonical path:', METSIM_HOURLY_PATH)
else:
    print('MetSim hourly output already at canonical path:', METSIM_HOURLY_PATH)

Wrote MetSim state file: /scratch/dlhogan/ess-project-data/domain_East_River_lumped/forcing/monthly_workflow/metsim/2020-01/metsim_state_2020-01.nc
Saved MetSim config: /scratch/dlhogan/ess-project-data/domain_East_River_lumped/forcing/monthly_workflow/metsim/2020-01/metsim_config_2020-01.ini
Configured start/stop: 2020-01-01 00:00:00 2020-01-31 00:00:00
Using domain file: /scratch/dlhogan/ess-project-data/domain_East_River_lumped/forcing/monthly_workflow/metsim/2020-01/metsim_domain_mask_2020-01.nc
MetSim command: /home/dlhogan/miniforge3/envs/metsim-run/bin/ms /scratch/dlhogan/ess-project-data/domain_East_River_lumped/forcing/monthly_workflow/metsim/2020-01/metsim_config_2020-01.ini
Most recent NetCDF in output dir: /scratch/dlhogan/ess-project-data/domain_East_River_lumped/forcing/monthly_workflow/metsim/2020-01/metsim_hourly_2020-01_20200101-20200131.nc
Copied latest NetCDF to canonical path: /scratch/dlhogan/ess-project-data/domain_East_River_lumped/forcing/monthly_workflow/metsim

In [44]:
# Step 7: Convert MetSim hourly output to SUMMA unit conventions and match reference schema.

def _find_var(ds, candidates, label):
    for name in candidates:
        if name in ds.data_vars:
            return name
    raise KeyError(f'Could not find {label}. Candidates: {candidates}. Available: {list(ds.data_vars)}')

def _to_kelvin(da):
    units = str(da.attrs.get('units', '')).lower()
    vals = da.values
    out = da.copy()
    if 'c' in units or 'degc' in units:
        if np.nanmedian(vals) < 170.0:
            out = da + 273.15
    elif 'k' in units:
        out = da.copy()
    else:
        out = da + 273.15 if np.nanmedian(vals) < 170.0 else da.copy()
    out.attrs['units'] = 'K'
    return out

def _to_flux_kg_m2_s(da):
    units = str(da.attrs.get('units', '')).lower()
    out = da.copy()
    if 'kg m-2 s-1' in units or 'kg m**-2 s**-1' in units:
        pass
    elif 'mm/s' in units or 'mm s-1' in units:
        pass
    elif 'mm timestep-1' in units or 'mm timestep^-1' in units:
        out = out / 3600.0
    elif 'mm/hr' in units or 'mm h-1' in units:
        out = out / 3600.0
    elif 'mm/day' in units or 'mm d-1' in units or 'mm day-1' in units:
        out = out / 86400.0
    else:
        out = out / 3600.0
    out.attrs['units'] = 'kg m**-2 s**-1'
    return out

def _to_w_per_m2_ref(da):
    units = str(da.attrs.get('units', '')).lower()
    out = da.copy()
    if 'w/m2' in units or 'w m-2' in units or 'w m**-2' in units:
        pass
    elif 'mj m-2 day-1' in units or 'mj/m2/day' in units:
        out = out * (1.0e6 / 86400.0)
    elif 'kj m-2 day-1' in units or 'kj/m2/day' in units:
        out = out * (1.0e3 / 86400.0)
    out.attrs['units'] = 'W m**-2'
    return out

def _to_pa(da):
    units = str(da.attrs.get('units', '')).lower()
    out = da.copy()
    if units in {'kpa', 'k pa'} or 'kpa' in units:
        out = out * 1000.0
    out.attrs['units'] = 'Pa'
    return out

def _to_kgkg(da):
    out = da.copy()
    vals = out.values
    if np.nanmedian(np.abs(vals)) > 1.0:
        out = out / 1000.0
    out.attrs['units'] = 'kg kg**-1'
    return out

def _to_wind_ref(da):
    out = da.copy()
    out.attrs['units'] = 'm s**-1'
    return out

def _sanity_check(ds_out):
    def _rng(name):
        arr = ds_out[name].values
        return float(np.nanmin(arr)), float(np.nanmax(arr)), float(np.nanmedian(arr))

    lo, hi, _ = _rng('airtemp')
    if lo < 180.0 or hi > 350.0:
        raise ValueError(f'airtemp out of plausible Kelvin range: min={lo}, max={hi}')

    lo, hi, _ = _rng('LWRadAtm')
    if lo < 0.0 or hi > 1000.0:
        raise ValueError(f'LWRadAtm out of plausible range (W m-2): min={lo}, max={hi}')

    lo, hi, _ = _rng('SWRadAtm')
    if lo < 0.0 or hi > 1400.0:
        raise ValueError(f'SWRadAtm out of plausible range (W m-2): min={lo}, max={hi}')

    lo, hi, _ = _rng('pptrate')
    if lo < 0.0 or hi > 0.02:
        raise ValueError(f'pptrate out of plausible range (kg m-2 s-1): min={lo}, max={hi}')

    lo, hi, _ = _rng('airpres')
    if lo < 50000.0 or hi > 110000.0:
        raise ValueError(f'airpres out of plausible range (Pa): min={lo}, max={hi}')

    lo, hi, _ = _rng('spechum')
    if lo < 0.0 or hi > 0.03:
        raise ValueError(f'spechum out of plausible range (kg kg-1): min={lo}, max={hi}')

def convert_metsim_to_summa_units(metsim_hourly_path, out_path, reference_path, domain_mask_path=None, apply_mask=True):
    with xr.open_dataset(metsim_hourly_path, engine='netcdf4', cache=False) as ds_in:
        ds = ds_in.load()
    with xr.open_dataset(reference_path, engine='netcdf4', cache=False) as ds_ref_in:
        ds_ref = ds_ref_in.load()

    air_name = _find_var(ds, ['airtemp', 'tair', 'temp', 'temperature'], 'air temperature')
    ppt_name = _find_var(ds, ['pptrate', 'precip', 'prec', 'rainf'], 'precipitation')
    sw_name = _find_var(ds, ['SWRadAtm', 'shortwave', 'swdown', 'srad'], 'shortwave radiation')
    lw_name = _find_var(ds, ['LWRadAtm', 'longwave', 'lwdown', 'lrad'], 'longwave radiation')
    ap_name = _find_var(ds, ['airpres', 'air_pressure', 'pressure'], 'air pressure')
    sh_name = _find_var(ds, ['spechum', 'specific_humidity', 'spec_humid'], 'specific humidity')
    ws_name = _find_var(ds, ['windspd', 'wind', 'wind_speed'], 'wind speed')

    ds_out = xr.Dataset()
    ds_out['airtemp'] = _to_kelvin(ds[air_name])
    ds_out['pptrate'] = _to_flux_kg_m2_s(ds[ppt_name])
    ds_out['SWRadAtm'] = _to_w_per_m2_ref(ds[sw_name])
    ds_out['LWRadAtm'] = _to_w_per_m2_ref(ds[lw_name])
    ds_out['airpres'] = _to_pa(ds[ap_name])
    ds_out['spechum'] = _to_kgkg(ds[sh_name])
    ds_out['windspd'] = _to_wind_ref(ds[ws_name])

    if apply_mask and domain_mask_path is not None and Path(domain_mask_path).exists():
        with xr.open_dataset(domain_mask_path, engine='netcdf4', cache=False) as dm:
            mname = 'mask' if 'mask' in dm.data_vars else list(dm.data_vars)[0]
            mask = dm[mname].load()
        ds_out = ds_out.where(mask > 0)
        ds_out.attrs['mask_applied'] = str(domain_mask_path)

    _sanity_check(ds_out)

    for cname in ['time', 'latitude', 'longitude']:
        if cname in ds_out.coords and cname in ds_ref.coords:
            ds_out[cname].attrs = dict(ds_ref[cname].attrs)

    for vname in ['airpres', 'LWRadAtm', 'SWRadAtm', 'pptrate', 'airtemp', 'spechum', 'windspd']:
        if vname in ds_ref.data_vars and vname in ds_out.data_vars:
            ref_attrs = dict(ds_ref[vname].attrs)
            ref_attrs['units'] = ds_out[vname].attrs.get('units', ref_attrs.get('units', ''))
            ds_out[vname].attrs = ref_attrs

    ordered_vars = [v for v in ds_ref.data_vars if v in ds_out.data_vars]
    ds_out = ds_out[ordered_vars].transpose('time', 'latitude', 'longitude')
    ds_out.attrs.update({k: v for k, v in ds_ref.attrs.items() if k in ['Conventions']})
    ds_out.attrs['source_reference_schema'] = str(reference_path)

    tmp_path = Path(out_path).with_name(f'{Path(out_path).stem}.tmp.nc')
    if tmp_path.exists():
        tmp_path.unlink()
    ds_out.to_netcdf(tmp_path)
    os.replace(tmp_path, out_path)
    return ds_out


if METSIM_HOURLY_PATH.exists():
    metsim_candidates = sorted(
        METSIM_DIR.glob(f'metsim_hourly_{MONTH}*.nc'),
        key=lambda p: p.stat().st_mtime,
        reverse=True,
    )
    metsim_input_path = metsim_candidates[0] if metsim_candidates else METSIM_HOURLY_PATH
    print('Using MetSim hourly input:', metsim_input_path)

    ds_summa = convert_metsim_to_summa_units(
        metsim_hourly_path=metsim_input_path,
        out_path=SUMMA_FORCING_PATH,
        reference_path=SUMMA_REFERENCE_PATH,
        domain_mask_path=METSIM_DOMAIN_PATH,
        apply_mask=APPLY_OUTPUT_MASK,
    )
    print('Wrote SUMMA-ready forcing:', SUMMA_FORCING_PATH)
    print('Vars:', list(ds_summa.data_vars))
    print('Units:', {v: ds_summa[v].attrs.get('units', '') for v in ds_summa.data_vars})
    print('Coords:', {c: ds_summa[c].attrs for c in ds_summa.coords})
else:
    print(f'MetSim hourly file not found yet: {METSIM_HOURLY_PATH}')
    print('Run MetSim first, or set METSIM_HOURLY_PATH to the correct output file.')

Using MetSim hourly input: /scratch/dlhogan/ess-project-data/domain_East_River_lumped/forcing/monthly_workflow/metsim/2020-01/metsim_hourly_2020-01_20200101-20200131.nc
Wrote SUMMA-ready forcing: /scratch/dlhogan/ess-project-data/domain_East_River_lumped/forcing/monthly_workflow/final_monthly/summa_forcing_2020-01.nc
Vars: ['airpres', 'LWRadAtm', 'SWRadAtm', 'pptrate', 'airtemp', 'spechum', 'windspd']
Units: {'airpres': 'Pa', 'LWRadAtm': 'W m**-2', 'SWRadAtm': 'W m**-2', 'pptrate': 'kg m**-2 s**-1', 'airtemp': 'K', 'spechum': 'kg kg**-1', 'windspd': 'm s**-1'}
Coords: {'time': {'long_name': 'time', 'standard_name': 'time'}, 'latitude': {'units': 'degrees_north', 'standard_name': 'latitude', 'long_name': 'latitude', 'stored_direction': 'decreasing'}, 'longitude': {'units': 'degrees_east', 'standard_name': 'longitude', 'long_name': 'longitude'}}


## Next Month Loop

To scale this to the full period, loop over month strings (for example `1999-10` to `2024-09`) and rerun the same stages with monthly file outputs. Then add a continuity check that verifies no missing or overlapping monthly files.